# Week 06 - Day 03: Snowflake Class Exercises (UK Travel Company)
**Dataset:** `uk_travel_bookings.csv` (30 bookings, 16 columns)  
**Revenue formula:** `travellers * price_per_person_gbp * (1 - COALESCE(discount_pct,0)/100)`

### Setup: Create Database, Warehouse, Table & Load CSV

In [1]:
CREATE DATABASE IF NOT EXISTS travel_db;
CREATE SCHEMA IF NOT EXISTS travel_db.bookings;

CREATE WAREHOUSE IF NOT EXISTS travel_wh
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

USE WAREHOUSE travel_wh;
USE SCHEMA travel_db.bookings;

CREATE OR REPLACE FILE FORMAT csv_ff
  TYPE = CSV
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  NULL_IF = ('');

CREATE OR REPLACE TABLE bookings (
  booking_id INT, booking_date DATE, customer_id STRING, customer_name STRING,
  home_city STRING, destination_city STRING, destination_country STRING,
  package_type STRING, travel_start_date DATE, nights INT, travellers INT,
  price_per_person_gbp NUMBER(10,2), discount_pct NUMBER(5,2),
  payment_mode STRING, status STRING, rating INT
);

CREATE OR REPLACE STAGE bookings_stage FILE_FORMAT = csv_ff;

COPY INTO bookings FROM @bookings_stage/uk_travel_bookings.csv ON_ERROR = 'CONTINUE';

FILE,STATUS,ROWS_PARSED,ROWS_LOADED
bookings_stage/uk_travel_bookings.csv,LOADED,30,30


### Level 1 — Q1. Show the first 10 bookings

In [2]:
SELECT * FROM bookings LIMIT 10;

BOOKING_ID,BOOKING_DATE,CUSTOMER_ID,CUSTOMER_NAME,HOME_CITY,DESTINATION_CITY,DESTINATION_COUNTRY,PACKAGE_TYPE,TRAVEL_START_DATE,NIGHTS,TRAVELLERS,PRICE_PER_PERSON_GBP,DISCOUNT_PCT,PAYMENT_MODE,STATUS,RATING
2001,2026-01-08,U001,Oliver Smith,London,Barcelona,Spain,City Break,2026-03-12,3,2,320,5.0,Credit Card,Completed,5.0
2002,2026-01-14,U002,Amelia Jones,Manchester,Faro,Portugal,Beach Holiday,2026-05-20,7,4,450,10.0,Debit Card,Completed,4.0
2003,2026-01-22,U003,Harry Williams,Birmingham,Rome,Italy,City Break,2026-04-02,4,2,410,null,Credit Card,Completed,5.0
2004,2026-02-03,U004,Isla Taylor,Edinburgh,Reykjavik,Iceland,Adventure,2026-03-28,5,2,890,10.0,Credit Card,Completed,5.0
2005,2026-02-09,U005,Jack Brown,Leeds,Palma,Spain,Beach Holiday,2026-06-10,7,3,520,0.0,Debit Card,Completed,3.0
2006,2026-02-15,U001,Oliver Smith,London,Paris,France,City Break,2026-04-18,2,2,280,5.0,Credit Card,Completed,4.0
2007,2026-02-21,U006,Sophie Davies,Cardiff,Santorini,Greece,Beach Holiday,2026-07-05,7,2,780,15.0,Credit Card,Completed,5.0
2008,2026-03-02,U007,George Evans,Bristol,Prague,Czech Republic,City Break,2026-05-08,3,4,290,null,Bank Transfer,Cancelled,null
2009,2026-03-10,U008,Poppy Wilson,Glasgow,Tenerife,Spain,Beach Holiday,2026-06-24,7,2,610,10.0,Debit Card,Completed,4.0
2010,2026-03-17,U009,Noah Thomas,London,Tokyo,Japan,Adventure,2026-09-14,10,2,1850,5.0,Credit Card,Completed,5.0


### Level 1 — Q2. How many bookings are there in total?

In [3]:
SELECT COUNT(*) AS total_bookings FROM bookings;

TOTAL_BOOKINGS
30


### Level 1 — Q3. List all bookings to Spain

In [4]:
SELECT booking_id, customer_name, destination_city, travel_start_date
FROM bookings
WHERE destination_country = 'Spain';

BOOKING_ID,CUSTOMER_NAME,DESTINATION_CITY,TRAVEL_START_DATE
2001,Oliver Smith,Barcelona,2026-03-12
2005,Jack Brown,Palma,2026-06-10
2009,Poppy Wilson,Tenerife,2026-06-24
2017,Freddie White,Malaga,2026-07-26
2022,George Evans,Valencia,2026-08-28
2029,Max Allen,Seville,2026-10-30


### Level 1 — Q4. Find all Cancelled bookings

In [5]:
SELECT booking_id, customer_name, destination_city, status
FROM bookings
WHERE status = 'Cancelled';

BOOKING_ID,CUSTOMER_NAME,DESTINATION_CITY,STATUS
2008,George Evans,Prague,Cancelled
2022,George Evans,Valencia,Cancelled


### Level 1 — Q5. Which bookings have no discount recorded (NULL)?

In [6]:
SELECT booking_id, customer_name, discount_pct
FROM bookings
WHERE discount_pct IS NULL;

BOOKING_ID,CUSTOMER_NAME,DISCOUNT_PCT
2003,Harry Williams,null
2008,George Evans,null
2013,Harry Williams,null
2020,Sophie Davies,null
2026,Freya Roberts,null
2030,Lily Hughes,null


### Level 2 — Q6. Calculate the revenue for every booking

In [7]:
SELECT booking_id,
       travellers * price_per_person_gbp * (1 - COALESCE(discount_pct,0)/100) AS revenue_gbp
FROM bookings;

BOOKING_ID,REVENUE_GBP
2001,608.0
2002,1620.0
2003,820.0
2004,1602.0
2005,1560.0
2006,532.0
2007,1326.0
2008,1160.0
2009,1098.0
2010,3515.0


### Level 2 — Q7. Total revenue by package type for non-cancelled bookings, highest first

In [8]:
SELECT package_type,
       ROUND(SUM(travellers * price_per_person_gbp * (1 - COALESCE(discount_pct,0)/100)), 2) AS revenue_gbp
FROM bookings
WHERE status <> 'Cancelled'
GROUP BY package_type
ORDER BY revenue_gbp DESC;

PACKAGE_TYPE,REVENUE_GBP
Beach Holiday,20665.0
Adventure,16591.0
City Break,8573.0


### Level 2 — Q8. Which destination country is booked most often?

In [9]:
SELECT destination_country, COUNT(*) AS bookings
FROM bookings
GROUP BY destination_country
ORDER BY bookings DESC
LIMIT 1;

DESTINATION_COUNTRY,BOOKINGS
Spain,6


### Level 2 — Q9. Share of bookings by payment mode

In [10]:
SELECT payment_mode,
       COUNT(*) AS bookings,
       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_share
FROM bookings
GROUP BY payment_mode
ORDER BY bookings DESC;

PAYMENT_MODE,BOOKINGS,PCT_SHARE
Credit Card,18,60.0
Debit Card,8,26.7
Bank Transfer,4,13.3


### Level 2 — Q10. Customers with more than 2 bookings

In [11]:
SELECT customer_name, COUNT(*) AS bookings
FROM bookings
GROUP BY customer_name
HAVING COUNT(*) > 2;

Query produced no results (0 rows)

### Level 2 — Q11. Average rating per destination country (ignore NULL ratings), with at least 2 rated trips

In [12]:
SELECT destination_country,
       ROUND(AVG(rating), 2) AS avg_rating,
       COUNT(rating) AS rated_trips
FROM bookings
GROUP BY destination_country
HAVING COUNT(rating) >= 2
ORDER BY avg_rating DESC;

DESTINATION_COUNTRY,AVG_RATING,RATED_TRIPS
France,4.50,2
Portugal,4.50,2
Greece,4.33,3
Spain,4.00,5


### Level 3 — Q12. Work out each trip's return date and how far in advance it was booked

In [13]:
SELECT booking_id, customer_name,
       DATEADD('day', nights, travel_start_date) AS return_date,
       DATEDIFF('day', booking_date, travel_start_date) AS lead_time_days
FROM bookings;

BOOKING_ID,CUSTOMER_NAME,RETURN_DATE,LEAD_TIME_DAYS
2001,Oliver Smith,2026-03-15,63
2002,Amelia Jones,2026-05-27,126
2003,Harry Williams,2026-04-06,70
2004,Isla Taylor,2026-04-02,53
2005,Jack Brown,2026-06-17,121
2006,Oliver Smith,2026-04-20,62
2007,Sophie Davies,2026-07-12,134
2008,George Evans,2026-05-11,67
2009,Poppy Wilson,2026-07-01,106
2010,Noah Thomas,2026-09-24,181


### Level 3 — Q13. Monthly bookings count and revenue by travel month

In [14]:
SELECT DATE_TRUNC('month', travel_start_date) AS travel_month,
       COUNT(*) AS bookings,
       ROUND(SUM(travellers * price_per_person_gbp * (1 - COALESCE(discount_pct,0)/100)), 2) AS revenue_gbp
FROM bookings
WHERE status <> 'Cancelled'
GROUP BY travel_month
ORDER BY travel_month;

TRAVEL_MONTH,BOOKINGS,REVENUE_GBP
2026-03-01,2,2210.0
2026-04-01,2,1352.0
2026-05-01,2,2220.0
2026-06-01,3,3398.0
2026-07-01,3,5502.0
2026-08-01,2,3642.0
2026-09-01,3,5832.0
2026-10-01,5,5120.0
2026-11-01,3,6538.0
2026-12-01,3,10015.0


### Level 3 — Q14. Rank customers by total spend within each home city

In [15]:
SELECT home_city, customer_name, total_spend,
       RANK() OVER (PARTITION BY home_city ORDER BY total_spend DESC) AS city_rank
FROM (
  SELECT home_city, customer_name,
         SUM(travellers * price_per_person_gbp * (1 - COALESCE(discount_pct,0)/100)) AS total_spend
  FROM bookings
  WHERE status <> 'Cancelled'
  GROUP BY home_city, customer_name
);

HOME_CITY,CUSTOMER_NAME,TOTAL_SPEND,CITY_RANK
Belfast,Ivy Clarke,4500.0,1
Birmingham,Rosie Hall,2760.0,1
Birmingham,Harry Williams,1560.0,2
Cardiff,Sophie Davies,2006.0,1
Edinburgh,Isla Taylor,2898.0,1
Glasgow,Theo Lewis,1728.0,1
Glasgow,Poppy Wilson,1098.0,2
Leeds,Jack Brown,2149.0,1
Leeds,Max Allen,594.0,2
Liverpool,Freya Roberts,2556.0,1


### Level 3 — Q15. For repeat customers, show the gap in days between consecutive bookings

In [16]:
SELECT customer_name, booking_date,
       DATEDIFF('day',
         LAG(booking_date) OVER (PARTITION BY customer_id ORDER BY booking_date),
         booking_date) AS days_since_last_booking
FROM bookings
QUALIFY COUNT(*) OVER (PARTITION BY customer_id) > 1
ORDER BY customer_name, booking_date;

CUSTOMER_NAME,BOOKING_DATE,DAYS_SINCE_LAST_BOOKING
Amelia Jones,2026-01-14,null
Amelia Jones,2026-03-25,70.0
Freya Roberts,2026-04-03,null
Freya Roberts,2026-07-25,113.0
George Evans,2026-03-02,null
George Evans,2026-06-22,112.0
Harry Williams,2026-01-22,null
Harry Williams,2026-04-11,79.0
Isla Taylor,2026-02-03,null
Isla Taylor,2026-04-27,83.0


### Level 3 — Q16. Top 2 destinations by revenue within each package type (use QUALIFY)

In [17]:
SELECT package_type, destination_city,
       SUM(travellers * price_per_person_gbp) AS revenue_gbp
FROM bookings
WHERE status <> 'Cancelled'
GROUP BY package_type, destination_city
QUALIFY ROW_NUMBER() OVER (PARTITION BY package_type ORDER BY SUM(travellers * price_per_person_gbp) DESC) <= 2;

PACKAGE_TYPE,DESTINATION_CITY,REVENUE_GBP
Adventure,Lapland,5000
Adventure,Bali,4200
Beach Holiday,Cancun,2900
Beach Holiday,Crete,2760
City Break,New York,2200
City Break,Rome,820


### Level 3 — Q17. Cancellation rate by package type

In [18]:
SELECT package_type,
       COUNT(*) AS bookings,
       SUM(IFF(status = 'Cancelled', 1, 0)) AS cancelled,
       ROUND(100 * SUM(IFF(status = 'Cancelled', 1, 0)) / COUNT(*), 1) AS cancel_rate_pct
FROM bookings
GROUP BY package_type;

PACKAGE_TYPE,BOOKINGS,CANCELLED,CANCEL_RATE_PCT
Adventure,6,0,0.0
Beach Holiday,11,0,0.0
City Break,13,2,15.4


### Level 3 — Q18. Segment bookings into Budget, Mid-range and Premium using CASE

In [19]:
SELECT booking_id, destination_city, price_per_person_gbp,
       CASE
         WHEN price_per_person_gbp < 400 THEN 'Budget'
         WHEN price_per_person_gbp < 900 THEN 'Mid-range'
         ELSE 'Premium'
       END AS price_band
FROM bookings;

BOOKING_ID,DESTINATION_CITY,PRICE_PER_PERSON_GBP,PRICE_BAND
2001,Barcelona,320,Budget
2002,Faro,450,Mid-range
2003,Rome,410,Mid-range
2004,Reykjavik,890,Mid-range
2005,Palma,520,Mid-range
2006,Paris,280,Budget
2007,Santorini,780,Mid-range
2008,Prague,290,Budget
2009,Tenerife,610,Mid-range
2010,Tokyo,1850,Premium
